# Lab L5.1 · Deploy the MIZAN fraud service — feature module, MLflow, model card, FastAPI and a CI gate
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 5 · ⏱ 90 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Package the Day 3 fraud model as a production-grade service: one feature module for training and serving, tracked experiments, a registered champion, a model card, a validated API, a load test and a release gate.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The card-authorisation platform team will integrate MIZAN fraud scoring next sprint. They need: a versioned package, an API contract, a threshold that produces ≈ 0.5% alerts, latency evidence, a model card for Model Risk, and an automated gate in the release pipeline.

**Progressive architecture:** the package you build is what Lab L5.2 monitors in production.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Move feature code into a module and prove parity

**Why this matters.** If training computes features in a notebook and serving re-implements them in Java, they *will* diverge — **training–serving skew**, a top cause of silent production failures. One module, imported by both, removes the risk. A **parity test** proves the module reproduces the training features exactly.

In [ ]:
import sys, json, importlib, joblib
SVC_DIR = CKPT / "mizan_fraud"                      # the deployable package lives next to the other checkpoints
sys.path.insert(0, str(CKPT))
SVC_DIR.mkdir(exist_ok=True); (SVC_DIR / "model").mkdir(exist_ok=True)
(SVC_DIR / "__init__.py").write_text('"""MIZAN fraud package."""\n')
print("package folder:", SVC_DIR)

In [ ]:
%%writefile ../../../checkpoints/mizan_fraud/features.py
"""mizan_fraud.features — the ONE place where fraud features are defined (training == serving)."""
import numpy as np
import pandas as pd


def build_fraud_features(tx: pd.DataFrame) -> pd.DataFrame:
    """Per-card behavioural (velocity) features using ONLY each card's past and present — safe for real-time scoring."""
    tx = tx.sort_values(["card_id", "txn_ts"]).reset_index(drop=True)
    g = tx.groupby("card_id")
    tx["secs_since_prev"] = g.txn_ts.diff().dt.total_seconds().fillna(30 * 86400)
    roll = tx.set_index("txn_ts").groupby("card_id").amount_aed
    tx["txn_count_1h"] = roll.rolling("1h").count().to_numpy()
    r24 = roll.rolling("24h").agg(["count", "sum"])
    tx["txn_count_24h"], tx["amount_sum_24h"] = r24["count"].to_numpy(), r24["sum"].to_numpy()
    prior_mean = g.amount_aed.transform(lambda s: s.shift(1).expanding().mean())
    tx["amount_vs_card_mean"] = tx.amount_aed / prior_mean.fillna(tx.amount_aed.median())
    tx["is_new_country_for_card"] = (~tx.duplicated(["card_id", "merchant_country"])).astype(int)
    tx["is_new_mcc_for_card"] = (~tx.duplicated(["card_id", "merchant_category"])).astype(int)
    tx["card_txn_index"] = g.cumcount()
    tx["hour"] = tx.txn_ts.dt.hour
    tx["is_night"] = tx.hour.between(0, 5).astype(int)
    tx["is_international"] = (tx.merchant_country != "AE").astype(int)
    tx["amount_to_salary"] = tx.amount_aed / tx.customer_monthly_salary_aed
    tx["log_amount"] = np.log1p(tx.amount_aed)
    return tx.sort_values("txn_ts").reset_index(drop=True)

NUMF = ["log_amount", "merchant_risk_score", "is_3ds_authenticated", "is_new_device", "distance_from_home_km", "customer_age",
        "card_tenure_months", "secs_since_prev", "txn_count_1h", "txn_count_24h", "amount_sum_24h", "amount_vs_card_mean",
        "is_new_country_for_card", "is_new_mcc_for_card", "card_txn_index", "hour", "is_night", "is_international", "amount_to_salary"]
CATF = ["merchant_category", "channel"]

In [ ]:
import mizan_fraud.features as F; importlib.reload(F)
raw = pd.read_csv(DATA / "card_transactions.csv.gz", parse_dates=["txn_ts"])
rebuilt = F.build_fraud_features(raw).set_index("txn_id")
ref = pd.read_parquet(CKPT / "card_features.parquet").set_index("txn_id")
ids = ref.sample(50_000, random_state=RNG).index
parity = {c: bool(np.allclose(rebuilt.loc[ids, c].astype(float), ref.loc[ids, c].astype(float), equal_nan=True)) for c in F.NUMF}
print("parity on all numeric features:", all(parity.values()))

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — features computed WITHOUT the card's earlier history

In [ ]:
sep_only = ____  # ✏️ B1
common = sep_only.index
for c in ["card_txn_index", "secs_since_prev", "is_new_country_for_card"]:
    print(f"{c:25s} differs on {(~np.isclose(sep_only[c], ref.loc[common, c])).mean():.1%} of September txns")

**✔ Checkpoint.** Parity `True`. With September-only data, most `card_txn_index` values and many `secs_since_prev` values differ — velocity features need each card’s **history**, which in production comes from a **feature store**.

## Step 2 · Track candidate models in MLflow

**Why this matters.** Three people tuning the same model in three notebooks is chaos. **Experiment tracking** records every candidate’s parameters, metrics and artefacts in one queryable store. We train three random-forest candidates on train, evaluate on validation at the 0.5% budget, and log everything.

#### ✏️ What to write in the next cell
- **Blank 2** *(inline)* — a database backend enables the model registry
- **Blank 3** *(inline)* — one tracked run per candidate
- **Blank 4** *(inline)* — record the hyperparameters

In [ ]:
from sklearn.metrics import average_precision_score
def at_budget(y, s, amount, budget=0.005) -> dict:
    """Ranking quality inside the fraud team's review budget (s = any score; higher = riskier)."""
    y, amount = np.asarray(y), np.asarray(amount)
    top = np.argsort(-np.asarray(s))[: int(len(y) * budget)]
    return {"precision@0.5%": y[top].mean(), "recall@0.5%": y[top].sum() / y.sum(),
            "AED_caught_share": (amount[top] * y[top]).sum() / (amount * y).sum(), "PR_AUC": average_precision_score(y, s)}
import os; os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
MLDIR = CKPT / "mlflow"; MLDIR.mkdir(exist_ok=True)
mlflow.set_tracking_uri(____)  # ✏️ B2
exp = mlflow.get_experiment_by_name("mizan-fraud") or mlflow.get_experiment(mlflow.create_experiment("mizan-fraud", artifact_location=(MLDIR / "artifacts").as_uri()))
mlflow.set_experiment(experiment_id=exp.experiment_id)
FEATS = F.NUMF + F.CATF
cf = ref.reset_index()
tr, va, te = (cf[cf.split == s] for s in ["train", "valid", "test"])
def make_pipe(**rf_params):
    prep = ColumnTransformer([("num", SimpleImputer(strategy="constant", fill_value=-1), F.NUMF),
                              ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), F.CATF)])
    return Pipeline([("prep", prep), ("rf", RandomForestClassifier(n_jobs=-1, random_state=RNG, **rf_params))])
tr_u = pd.concat([tr[tr.is_fraud == 1], tr[tr.is_fraud == 0].sample(250_000, random_state=RNG)])
CONFIGS = [dict(n_estimators=150, min_samples_leaf=25, max_features="sqrt"),
           dict(n_estimators=300, min_samples_leaf=10, max_features="sqrt"),
           dict(n_estimators=300, min_samples_leaf=5, max_features=0.5)]
for cfg in CONFIGS:
    with ____:  # ✏️ B3
        t0 = time.perf_counter(); m = make_pipe(**cfg).fit(tr_u[FEATS], tr_u.is_fraud); fit_s = time.perf_counter() - t0
        met = at_budget(va.is_fraud, m.predict_proba(va[FEATS])[:, 1], va.amount_aed)
        ____  # ✏️ B4
        mlflow.log_metrics({k.replace("@", "_at_").replace("%", "pct"): float(v) for k, v in met.items()} | {"fit_seconds": fit_s})
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], filter_string="tags.mlflow.runName LIKE 'rf-%'", order_by=["metrics.PR_AUC DESC"])
display(runs[["run_id", "tags.mlflow.runName", "metrics.PR_AUC", "metrics.recall_at_0.5pct", "metrics.fit_seconds"]].head(6))
best_run = runs.iloc[0]; best_cfg = next(c for c in CONFIGS if f"rf-{c['n_estimators']}-{c['min_samples_leaf']}-{c['max_features']}" == best_run["tags.mlflow.runName"])
print("best config:", best_cfg)

**✔ Checkpoint.** Three runs logged; PR-AUC differences are small (≈ 0.94 ± 0.01) but training times differ a lot — a real trade-off to document.

## Step 3 · Refit, set the alert threshold, write the model card, register the champion

**Why this matters.** The chosen configuration is refit on train **+ valid** (more, and more recent, data). The **alert threshold** is set so that ≈ 0.5% of validation traffic alerts — the operational budget. A **model card** documents intended use, data, performance, limitations and owners; Model Risk will not approve a model without one. The registry alias `champion` is what deployment uses.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — alert on the riskiest 0.5%
- **Blank 6** *(inline)* — deployment asks for 'champion', never a file path

In [ ]:
cand = make_pipe(**best_cfg).fit(tr_u[FEATS], tr_u.is_fraud)
threshold = float(np.quantile(cand.predict_proba(va[FEATS])[:, 1], ____))  # ✏️ B5
trva = pd.concat([tr, va]); trva_u = pd.concat([trva[trva.is_fraud == 1], trva[trva.is_fraud == 0].sample(300_000, random_state=RNG)])
final = make_pipe(**best_cfg).fit(trva_u[FEATS], trva_u.is_fraud)
test_met = {k: float(v) for k, v in at_budget(te.is_fraud, final.predict_proba(te[FEATS])[:, 1], te.amount_aed).items()}
te_scores = final.predict_proba(te[FEATS])[:, 1]
alert_rate = float((te_scores >= threshold).mean())
recall_at_thr = float((te_scores[te.is_fraud.to_numpy() == 1] >= threshold).mean())
card = {
    "name": "mizan-fraud", "version": "1.0.0", "owner": "MIZAN squad — Fraud Analytics", "approver": "Model Risk Management",
    "intended_use": "Rank card transactions for manual fraud review (budget ≈ 0.5% of traffic). NOT for automatic card blocking.",
    "training_data": "card_transactions Apr–Aug 2025 (undersampled legit 300k + all fraud)", "features": FEATS,
    "algorithm": f"RandomForestClassifier {best_cfg}", "alert_threshold": threshold, "test_metrics_sep_2025": test_met, "test_alert_rate": alert_rate, "test_recall_at_threshold": recall_at_thr,
    "limitations": ["Velocity features require card history from the feature store", "Probabilities are biased by undersampling — use for ranking",
                    "Not validated on channels absent from training (e.g. Digital_Wallet)", "Fraud patterns drift — monitor weekly (Lab L5.2)"],
    "fairness": "No demographic inputs; monitor alert rates by customer segment", "retrain_policy": "Monthly or on PSI > 0.25 / recall drop > 10 pts",
}
joblib.dump(final, SVC_DIR / "model" / "fraud_rf.joblib", compress=3)
(SVC_DIR / "model" / "model_card.json").write_text(json.dumps(card, indent=2))
with mlflow.start_run(run_name="champion-refit-train+valid") as run:
    mlflow.log_params(best_cfg | {"train_rows": len(trva_u)}); mlflow.log_metrics({"test_" + k.replace("@", "_at_").replace("%", "pct"): v for k, v in test_met.items()})
    mlflow.log_artifacts(str(SVC_DIR / "model"), artifact_path="model")
mc = MlflowClient()
if not mc.search_registered_models(filter_string="name='mizan-fraud'"):
    mc.create_registered_model("mizan-fraud", description="MIZAN card-fraud ranking model")
mv = mc.create_model_version("mizan-fraud", f"{run.info.artifact_uri}/model", run.info.run_id)
____  # ✏️ B6
print(f"registered mizan-fraud v{mv.version} as champion | threshold {threshold:.4f} | test alert rate {alert_rate:.2%}"); print(test_met)

**✔ Checkpoint.** Threshold ≈ 0.3–0.6 (undersampled scores); test alert rate ≈ 0.4–0.6%; test recall@0.5% ≈ 0.9.

## Step 4 · Serve it: FastAPI with Pydantic validation

**Why this matters.** The authorisation platform calls a **contract**, not a notebook. Pydantic enforces types and ranges and an allow-list of channels; malformed or hostile input never reaches the model. `/health` exposes the live version for observability.

In [ ]:
%%writefile ../../../checkpoints/mizan_fraud/app.py
"""mizan_fraud.app — real-time fraud scoring service (FastAPI)."""
import json, time
from pathlib import Path
from typing import List, Literal, Optional
import joblib, numpy as np, pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

HERE = Path(__file__).parent
MODEL = joblib.load(HERE / "model" / "fraud_rf.joblib")
CARD = json.loads((HERE / "model" / "model_card.json").read_text())
THRESHOLD = CARD["alert_threshold"]

class Txn(BaseModel):
    """Payload = raw transaction fields + velocity features served by the feature store."""
    txn_id: str
    amount_aed: float = Field(gt=0, lt=2_000_000)
    merchant_category: str
    channel: Literal["POS_Chip", "Contactless", "E-Commerce", "ATM", "Digital_Wallet"]
    merchant_risk_score: Optional[float] = Field(default=None, ge=0, le=1)
    is_3ds_authenticated: Optional[float] = None
    is_new_device: Optional[float] = None
    distance_from_home_km: Optional[float] = Field(default=None, ge=0)
    customer_age: int = Field(ge=18, le=100)
    card_tenure_months: int = Field(ge=0)
    secs_since_prev: float = Field(ge=0)
    txn_count_1h: int = Field(ge=1)
    txn_count_24h: int = Field(ge=1)
    amount_sum_24h: float = Field(ge=0)
    amount_vs_card_mean: float = Field(ge=0)
    is_new_country_for_card: int = Field(ge=0, le=1)
    is_new_mcc_for_card: int = Field(ge=0, le=1)
    card_txn_index: int = Field(ge=0)
    hour: int = Field(ge=0, le=23)
    is_night: int = Field(ge=0, le=1)
    is_international: int = Field(ge=0, le=1)
    amount_to_salary: float = Field(ge=0)

app = FastAPI(title="MIZAN fraud scoring", version=CARD["version"])

def _score(rows: List[Txn]):
    df = pd.DataFrame([r.model_dump() for r in rows])
    df["log_amount"] = np.log1p(df.amount_aed)
    df = df.astype({c: float for c in ["merchant_risk_score", "is_3ds_authenticated", "is_new_device", "distance_from_home_km"]})
    s = MODEL.predict_proba(df[CARD["features"]])[:, 1]
    return [{"txn_id": r.txn_id, "fraud_score": round(float(p), 5), "alert": bool(p >= THRESHOLD),
             "model_version": CARD["version"]} for r, p in zip(rows, s)]

@app.get("/health")
def health():
    return {"status": "ok", "model": CARD["name"], "version": CARD["version"], "threshold": THRESHOLD}

@app.post("/score")
def score(txn: Txn):
    t0 = time.perf_counter(); out = _score([txn])[0]
    out["latency_ms"] = round((time.perf_counter() - t0) * 1000, 2); return out

@app.post("/score_batch")
def score_batch(txns: List[Txn]):
    return _score(txns)

#### ✏️ What to write in the next cell
- **Blank 7** *(inline)* — in-process HTTP client — no server needed
- **Blank 8** *(inline)* — expect 422 Unprocessable Entity

In [ ]:
import mizan_fraud.app as svc; importlib.reload(svc)
from fastapi.testclient import TestClient
client = ____  # ✏️ B7
print(client.get("/health").json())

def payload(row: pd.Series) -> dict:
    d = {k: row[k] for k in ["txn_id", "amount_aed"] + [f for f in FEATS if f != "log_amount"]}
    return {k: (None if isinstance(v, float) and np.isnan(v) else (v.item() if hasattr(v, "item") else v)) for k, v in d.items()}
sample = pd.concat([te[te.is_fraud == 1].head(1), te[te.is_fraud == 0].head(2)])
for _, r in sample.iterrows():
    resp = client.post("/score", json=payload(r)); print(resp.status_code, resp.json(), "| truth:", r.is_fraud)
good = payload(sample.iloc[0])
for bad in [good | {"amount_aed": -50}, good | {"channel": "Crypto_ATM"}]:
    resp = client.post("/score", json=bad)
    print("invalid →", ____, [e["loc"][-1] for e in resp.json()["detail"]])  # ✏️ B8

**✔ Checkpoint.** `/health` shows v1.0.0; the fraud row gets a high score and `alert: true`; invalid payloads return 422 naming the bad field.

## Step 5 · Load test: latency and throughput

**Why this matters.** An authorisation SLA is typically **p95 < 50–100 ms end-to-end**. We measure the service’s latency distribution on 300 single calls and the throughput of `/score_batch` on 5,000 transactions. Averages hide tail latency — always report percentiles.

In [ ]:
rows = [payload(r) for _, r in te.sample(300, random_state=RNG).iterrows()]
lat = []
for p in rows:
    t0 = time.perf_counter(); client.post("/score", json=p); lat.append((time.perf_counter() - t0) * 1000)
lat = np.array(lat)
print(f"single /score: p50 {np.percentile(lat, 50):.1f} ms | p95 {np.percentile(lat, 95):.1f} ms | p99 {np.percentile(lat, 99):.1f} ms")
batch = [payload(r) for _, r in te.sample(5_000, random_state=1).iterrows()]
t0 = time.perf_counter(); out = client.post("/score_batch", json=batch).json(); dt = time.perf_counter() - t0
print(f"/score_batch: {len(out):,} txns in {dt:.2f}s → {len(out) / dt:,.0f} txns/s")
plt.hist(lat, bins=40); plt.axvline(np.percentile(lat, 95), color="r", label="p95"); plt.xlabel("ms"); plt.legend(); plt.title("Single-request latency"); plt.show()

**✔ Checkpoint.** Single-call p95 in the tens of ms on a laptop (RF with `n_jobs=-1` per call adds overhead); batch throughput thousands of txns/s.

## Step 6 · Nightly batch job on new raw data + CI release gate

**Why this matters.** Not everything is real-time: Fraud Ops also wants a **nightly batch** report on the previous day. The batch job must use the same feature module with **history warm-up**. Finally a **release gate** encodes our standards as code; its JSON output is the evidence attached to the change ticket.

#### ✏️ What to write in the next cell
- **Blank 9** *(code block, ≈2 lines)* — build features on history + new week together (warm-up), then keep only the new week's rows (txn_id in week)

In [ ]:
prod = pd.read_csv(DATA / "card_transactions_prod_2025Q4.csv.gz", parse_dates=["txn_ts"])
week = prod[prod.txn_ts < "2025-10-08"]
hist = raw[raw.txn_ts >= "2025-08-01"]
# ✏️ BLANK 9 — write ≈2 line(s): build features on history + new week together (warm-up), then keep only the new week's rows (txn_id in week)
# ...your code here...
scored["fraud_score"] = final.predict_proba(scored[FEATS])[:, 1]
scored["alert"] = scored.fraud_score >= threshold
daily = scored.groupby(scored.txn_ts.dt.date).agg(txns=("txn_id", "size"), alerts=("alert", "sum"), fraud=("is_fraud", "sum"))
daily["alert_rate"] = daily.alerts / daily.txns; display(daily)
scored[["txn_id", "txn_ts", "card_id", "amount_aed", "fraud_score", "alert"]].to_parquet(CKPT / "batch_scores_2025-10-01_07.parquet", index=False)

In [ ]:
def release_gate(model, card) -> dict:
    checks = {"test PR-AUC >= 0.85": card["test_metrics_sep_2025"]["PR_AUC"] >= 0.85,
              "0.3% <= test alert rate <= 0.8%": 0.003 <= card["test_alert_rate"] <= 0.008,
              "p95 latency < 150 ms": float(np.percentile(lat, 95)) < 150,
              "no ID / PII features": not {"txn_id", "card_id", "customer_id"} & set(card["features"]),
              "threshold in (0, 1)": 0 < card["alert_threshold"] < 1}
    return {k: bool(v) for k, v in checks.items()}
gate = release_gate(final, card)
(CKPT / "release_gate.json").write_text(json.dumps({"model": "mizan-fraud", "version": card["version"], "checks": gate,
                                                    "decision": "DEPLOY" if all(gate.values()) else "BLOCK"}, indent=2))
print(json.dumps(gate, indent=2)); print("DEPLOY" if all(gate.values()) else "BLOCK")

**✔ Checkpoint.** Daily alert rates wander between ≈ 0.3% and 0.6%. Compare `alerts` with `fraud`: on several days the model alerts far fewer transactions than there are frauds — an early hint of trouble (new Digital_Wallet channel, new scam typology). All gate checks pass → DEPLOY.

## ✅ Lab complete
MIZAN fraud v1.0.0 is packaged, registered, documented, served and gated. **Next:** keep it healthy in production.